[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA3/blob/main/en/lab9/lab9.ipynb)

# Practical 9: One Class Network

## Pre-requisites

### Install packages

If the practical requires any Python package, a cell must be included where they are installed. If we use a package that has been used in previous practicals, we could take for granted that it is installed, but it costs nothing to satisfy all the dependencies in the practical itself to reduce the dependencies between them.

In [ ]:
import torch
from torchvision import datasets

# We make the necessary imports
import numpy as np

# One Class on artificial data

The first thing we have to do is define the data to use.

In [ ]:
random_state = 42
rng = np.random.RandomState(random_state)
#  training data
X = 0.3 * rng.randn(5000, 2)
x_train = np.r_[X + 2, X - 2]
#  test data in the same distribution as the training data
X = 0.3 * rng.randn(200, 2)
x_test = np.r_[X + 2, X - 2]
#  outliers
x_outliers = rng.uniform(low=-4, high=4, size=(200, 2))

## Create your own network for anomaly detection

We are going to create our own network for anomaly detection. To do this, we will use a network that transforms each input element into a numerical value. We will optimize its weights so that:
* They are small (L2 regularization)
* The output is greater than a value `r` that we will modify in each epoch.
In each epoch, we will calculate `r` so that only a small fraction of the data $\nu$ gets an output $\tilde{y}$ lower than `r`. In this way, after several epochs, the anomalous inputs will be those that do not exceed that value ($\tilde{y} <= r$).

The idea is that these two opposing optimization objectives (L2 is minimized by bringing the weights - and, therefore, the output - to zero; however, the other loss increases when the output does not reach `r`) cause the weights that make the output reach `r` to be assigned to the most frequent patterns. When an anomalous data point is introduced, it will not reach `r`.

We will define any network that **transforms the input data into a single-element output**. This network will fulfill a series of characteristics:

* The layer before the output will be the so-called **deep features**.
* All the layers (including the last one) must include regularization.
* The cost function is $$L(y, \tilde{y}) = \dfrac{1}{2} \| w^2 \| + \dfrac{1}{\nu} \dfrac{1}{N} \sum_{i=1}^N \max(0, r - \tilde{y}) $$ where $\tilde{y}$ is the output of the network, $\nu$ is a hyperparameter between 0 and 1, and $r$ is a non-trainable parameter, but it is going to be modified in each epoch.
* At the end of each epoch, r is going to be modified to the value of the $\nu$-quantile of the input data (this value will be modified thanks to the Callback provided below).
* For the prediction, a typical data point will be considered if $\tilde{y} > r$. Otherwise, it will be an atypical data point.

In [ ]:
class ChangeRCallback:
    def __init__(self, train_data, delta=.025, steps=3):
        self.train_data = train_data
        self.delta = delta
        self.steps = steps
        self.count = 0
        self.stop_training = False # With this we will indicate to the training loop when to stop

    def on_epoch_end(self, model):
        model.eval()
        with torch.no_grad():
            preds = model(self.train_data).view(-1)

        sorted_values, _ = torch.sort(preds)
        idx = int(len(sorted_values) * (1. - model.nu))
        new_value = sorted_values[idx].item()

        old_value = model.r.item()

        print('Changing r to', new_value,
              ', max:', sorted_values.max().item(),
              ', min:', sorted_values.min().item())

        # update r
        model.r.data = torch.tensor(new_value, device=model.r.device)

        if abs(old_value - new_value) < self.delta:
            self.count += 1
            if self.count >= self.steps:
                print('Convergence reached. Ending the training.')
                self.stop_training = True
        else:
            self.count = 0

Your job is to create the model and train it.

In [ ]:
import torch.nn as nn
import torch.optim as optim

# TODO: implement the anomaly detection network

class AnomalyDetector:

    def __init__(self, input_shape, nu=.5):
        # TODO : define the model
        self.model = None

        # Parameters (not trainable)
        self.r = torch.tensor(1.0, requires_grad=False)
        self.nu = torch.tensor(nu, requires_grad=False)

        # We move everything to device (cpu/cuda)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if self.model is not None:
            self.model.to(self.device)
            self.r = self.r.to(self.device)
            self.nu = self.nu.to(self.device)

        # TODO: create the optimizer
        self.optimizer = None

        # TODO: define the loss function
        self.criterion = None

    def loss_function(self, y_true, y_pred):
        # TODO: create the loss function
        pass

    def fit(self, X, y=None, sample_weight=None):
        # TODO: train the model (manual loop of epochs and batches)
        # remember to use:
        # - self.model.train()
        # - zero_grad(), backward(), step()
        # - ChangeRCallback class defined above
        pass

    def predict(self, X):
        # TODO: Return the prediction of the model
        # use model.eval() and torch.no_grad()
        pass

    def __del__(self):
        # TODO: delete the model
        if hasattr(self, "model"):
            del self.model
        torch.cuda.empty_cache()  # free GPU memory

### Train the model.

Use what was done before to train your model.

In [ ]:
# TODO: Define the model

In [ ]:
# TODO: Train your model

## Evaluating the model

Once trained, to evaluate the model we only have to take into account the following:

  1. If the output is greater than r, it is a typical data point.
  1. If the output is less than r, it is an atypical data point.

### TASK: Evaluate the model with the test set data, and with the outliers. Visualize the typical and atypical data with a plot.

In [ ]:
# TODO: Evaluate the model with the test set data. Indicate the percentage of data labeled as typical, and visualize the data

In [ ]:
# TODO: Evaluate the model with the outlier data. Indicate the percentage of data labeled as atypical, and visualize the data together with the test ones

What results have you obtained? If the number of detected outliers is low (below 30%), you may be making some mistake, among them:

* Overfitting the model. Try using a different delta in the callback.
* Using a too high value of $\nu$.

Try different configurations to see their effect.

# CONGRATULATIONS! You have completed the one class practical.